# 03 - Modelagem, IA e Sistemas Inteligentes

Notebook responsável pelas análises avançadas, modelagem estatística, inteligência artificial, sistemas de recomendação e experimentos de IA generativa aplicados aos dados de compras.

Objetivos:
- Market Basket Analysis
- Sistemas de Recomendação
- Embeddings de Produtos
- IA Generativa
- Insights Automáticos
- Modelagem Temporal

In [1]:
# !pip install -r requirements.txt

In [2]:
# IMPORTS
import time
from pathlib import Path
import pandas as pd
import warnings

from src.core.tictoc import tictoc
from src.core.config_loader import ConfigLoader
from src.core.paths import ROOT_DIR
from src.core.logger import setup_logger
from src.core.model import *

# CONFIGURAÇÃO
warnings.filterwarnings('ignore')
logger = setup_logger(__name__)
tic_geral = time.time()

config = ConfigLoader(
    "config/settings.yaml"
)

2026-11-17 01:13:18.285 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


## 1. Carregamento dos Dados

Nesta etapa é realizado o carregamento da base analítica consolidada utilizada nos experimentos de modelagem estatística, inteligência artificial, sistemas de recomendação e análises avançadas de comportamento de consumo.

| Variável             | Tipo                | Descrição                             |
| -------------------- | ------------------- | ------------------------------------- |
| chave_anonimizada    | categórica          | Identificador único da nota fiscal    |
| data_hora            | datetime            | Data e hora da compra                 |
| mes_ano              | categórica temporal | Ano e mês da compra                   |
| dia_semana           | categórica          | Dia da semana da compra               |
| feriado              | binária             | Indica se a compra ocorreu em feriado |
| estacao_ano          | categórica          | Estação climática                     |
| temperatura_max      | numérica contínua   | Temperatura máxima do dia             |
| temperatura_min      | numérica contínua   | Temperatura mínima do dia             |
| temperatura_media    | numérica contínua   | Temperatura média do dia              |
| chuva_mm             | numérica contínua   | Volume de chuva                       |
| cat_temperatura      | categórica ordinal  | Faixa de temperatura                  |
| dia_chuvoso          | binária             | Indica ocorrência de chuva            |
| periodo_dia          | categórica ordinal  | Madrugada, manhã, tarde ou noite      |
| cnpj                 | categórica          | Identificador do supermercado         |
| supermercado         | categórica          | Nome do supermercado                  |
| qtd_total_nota       | numérica contínua   | Quantidade total de produtos da nota  |
| valor_total_nota     | numérica contínua   | Valor total da nota fiscal            |
| valor_total_tributos | numérica contínua   | Soma dos tributos da nota             |
| cod_produto          | categórica          | Código do produto                     |
| categoria_produto    | categórica          | Categoria do produto                  |
| produto              | textual/categórica  | Nome do produto                       |
| unidade              | categórica          | Unidade de venda                      |
| quantidade           | numérica contínua   | Quantidade comprada                   |
| preco_unitario       | numérica contínua   | Preço unitário                        |
| preco_total          | numérica contínua   | Valor total do item                   |

In [3]:
tic = time.time()

In [4]:
data_path = config.get(
    "paths.data"
)
df = pd.read_excel(data_path)
print(
    f"Linhas: {df.shape[0]}"
)
print(
    f"Colunas: {df.shape[1]}"
)
df.head()

Linhas: 4120
Colunas: 25


,chave_anonimizada,data_hora,mes_ano,dia_semana,feriado,estacao_ano,temperatura_max,temperatura_min,temperatura_media,chuva_mm,...,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,categoria_produto,produto,unidade,quantidade,preco_unitario,preco_total
0,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,81230001,CEREAL,ALHO A GRANEL kg,KG,0.218,14.90,3.24
1,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11678230001,UTILIDADES,PINO T 3 SAIDAS IL,UNIDADE,1.000,9.20,9.20
2,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,262911,MOLHO,MILHO QUERO 170G,UNIDADE,1.000,3.39,3.39
3,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11421800001,MERCEARIA,BEB VERY 1250G SALAD,UNIDADE,1.000,7.70,7.70
4,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,18730001,MERCEARIA,ARR T JOAO LF T1 5kg,UNIDADE,1.000,25.90,25.90


In [5]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:00:02) [17/11/2026 01:13:18 - 17/11/2026 01:13:20]


### 2. Entendimento Inicial dos Dados

**POSSÍVEIS VARIÁVEIS RESPOSTA - MODELOS SUPERVISIONADOS**

**Regressão**
| variável resposta | objetivo              |
| ----------------- | --------------------- |
| preco_unitario    | previsão de preço     |
| preco_total       | previsão de gasto     |
| valor_total_nota  | previsão de ticket    |
| quantidade        | previsão de consumo   |
| chuva_mm          | inferência contextual |

**Classificação**
| variável resposta | objetivo                                            |
| ----------------- | --------------------------------------------------- |
| feriado           | prever padrão feriado                               |
| categoria_produto | classificação textual                               |
| produto           | prever a compra de determinado produto (ex: Whisky) |

---

**MODELOS NÃO SUPERVISIONADOS**

**Clusterização**
| técnica | objetivo            |
| ------- | ------------------- |
| KMeans  | tipos de compra     |
| HDBSCAN | padrões raros       |
| PCA     | redução dimensional |
| UMAP    | embeddings visuais  |

---

**ASSOCIATION RULES**

| técnica   | objetivo                  |
| --------- | ------------------------- |
| FP-Growth | produtos comprados juntos |
| Apriori   | cesta                     |
| Lift      | relações fortes           |

---

**NLP / IA**

| técnica               | objetivo             |
| --------------------- | -------------------- |
| Embeddings            | similaridade         |
| LLM                   | insights automáticos |
| RAG                   | consulta inteligente |
| Recommendation System | recomendação         |

In [6]:
tic = time.time()

In [7]:
results = analyze_dataset(df)

2026-11-17 01:13:20,927 | src.core.model | INFO | Iniciando análise exploratória da base
2026-11-17 01:13:21,053 | src.core.model | INFO | Análise exploratória concluída


#### 2.1 Shape

In [8]:
results['shape']

{'linhas': 4120, 'colunas': 25}

#### 2.2 Nulos

In [9]:
results['missing']

,total_nulos,percentual_nulos
valor_total_tributos,27,0.66
data_hora,0,0.00
mes_ano,0,0.00
dia_semana,0,0.00
feriado,0,0.00
estacao_ano,0,0.00
temperatura_max,0,0.00
temperatura_min,0,0.00
chave_anonimizada,0,0.00
temperatura_media,0,0.00


#### 2.3 Estatísticas Numéricas

In [10]:
results['numeric_description']

,count,mean,std,min,25%,50%,75%,max
temperatura_max,4120.0,2.245163e+01,3.896148e+00,11.700,20.60,22.50,25.40,3.240000e+01
temperatura_min,4120.0,1.391405e+01,3.434759e+00,2.200,10.80,13.90,16.80,2.070000e+01
temperatura_media,4120.0,1.748175e+01,3.373459e+00,7.300,15.30,17.30,20.30,2.620000e+01
chuva_mm,4120.0,3.690437e+00,6.688817e+00,0.000,0.00,1.00,4.10,6.310000e+01
qtd_total_nota,4120.0,4.448010e+01,3.371493e+01,1.000,18.00,33.00,71.00,1.180000e+02
valor_total_nota,4120.0,3.806797e+02,2.959953e+02,3.490,140.33,268.60,648.32,9.995600e+02
valor_total_tributos,4093.0,1.020302e+02,9.454931e+01,0.000,18.15,76.32,169.39,3.234700e+02
cod_produto,4120.0,1.145790e+09,3.319579e+09,74.000,98558.00,267042.00,1197990.00,1.204739e+10
quantidade,4120.0,1.268575e+00,1.448860e+00,0.035,1.00,1.00,1.00,1.400000e+01
preco_unitario,4120.0,8.301201e+00,8.896714e+00,0.690,3.39,5.99,9.19,1.499000e+02


#### 2.4 Cardinalidade

In [11]:
results['cardinality']

,variavel,qtd_unicos
20,produto,1292
18,cod_produto,1292
24,preco_total,781
23,preco_unitario,505
0,chave_anonimizada,279
1,data_hora,279
16,valor_total_nota,250
22,quantidade,240
17,valor_total_tributos,233
6,temperatura_max,121


#### 2.5 Correlação

In [12]:
results['correlation']

,temperatura_max,temperatura_min,temperatura_media,chuva_mm,dia_chuvoso,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,quantidade,preco_unitario,preco_total
temperatura_max,1.000000,0.749939,0.919932,-0.022684,-0.025551,-0.197539,-0.085985,-0.048378,0.067190,0.047651,0.008818,0.043728
temperatura_min,0.749939,1.000000,0.929710,0.268224,0.403509,-0.224198,-0.122571,-0.066328,0.000520,0.051024,0.000499,0.035675
temperatura_media,0.919932,0.929710,1.000000,0.151191,0.222979,-0.250268,-0.125747,-0.077849,0.043312,0.052715,0.008190,0.047296
chuva_mm,-0.022684,0.268224,0.151191,1.000000,0.322087,-0.123268,-0.065921,-0.035777,-0.043633,0.028841,0.015696,0.035583
dia_chuvoso,-0.025551,0.403509,0.222979,0.322087,1.000000,-0.090302,-0.039699,-0.003028,-0.069782,0.038582,-0.022387,0.002545
qtd_total_nota,-0.197539,-0.224198,-0.250268,-0.123268,-0.090302,1.000000,0.934965,0.895038,-0.161558,-0.021629,-0.036154,-0.041345
valor_total_nota,-0.085985,-0.122571,-0.125747,-0.065921,-0.039699,0.934965,1.000000,0.954783,-0.184678,0.019486,0.017073,0.036229
valor_total_tributos,-0.048378,-0.066328,-0.077849,-0.035777,-0.003028,0.895038,0.954783,1.000000,-0.287827,0.008196,0.021267,0.037445
cod_produto,0.067190,0.000520,0.043312,-0.043633,-0.069782,-0.161558,-0.184678,-0.287827,1.000000,0.054134,-0.036307,-0.017512
quantidade,0.047651,0.051024,0.052715,0.028841,0.038582,-0.021629,0.019486,0.008196,0.054134,1.000000,-0.126128,0.251416


#### 2.7 Outliers

In [13]:
pd.DataFrame(results['outliers']).T

,total_outliers,percentual_outliers
temperatura_max,50.0,1.21
temperatura_min,0.0,0.00
temperatura_media,6.0,0.15
chuva_mm,478.0,11.60
qtd_total_nota,0.0,0.00
valor_total_nota,0.0,0.00
valor_total_tributos,0.0,0.00
cod_produto,862.0,20.92
quantidade,683.0,16.58
preco_unitario,393.0,9.54


#### 2.8 Categorias mais frequentes

In [14]:
results['categorical_summary']['categoria_produto']

categoria_produto
BEBIDA NAO ALCOOLICA    1435
MERCEARIA                473
PADARIA E FRIOS          307
CONGELADO                244
BOMBONIERE               231
MASSA                    226
HIGIENE E LIMPEZA        202
BEBIDA ALCOOLICA         171
MOLHO                    169
HORTIFRUTI               122
Name: count, dtype: int64

#### 2.9 Intervalo Temporal

In [15]:
results['date_range']

{'data_min': Timestamp('2022-06-28 20:44:50'),
 'data_max': Timestamp('2026-11-16 00:25:34'),
 'dias': 1601}

In [16]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:00) [17/11/2026 01:13:20 - 17/11/2026 01:13:21]


### 3. Feature Engineering

In [17]:
tic = time.time()

In [18]:
# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

df_model = feature_engineering(df)

print(
    f'Colunas originais: {df.shape[1]}'
)

print(
    f'Colunas após FE: {df_model.shape[1]}'
)

2026-11-17 01:13:21,317 | src.core.model | INFO | Iniciando feature engineering
2026-11-17 01:13:21,329 | src.core.model | INFO | Criando features temporais
2026-11-17 01:13:21,346 | src.core.model | INFO | Criando features financeiras
2026-11-17 01:13:21,352 | src.core.model | INFO | Criando features de frequência
2026-11-17 01:13:21,364 | src.core.model | INFO | Criando features de preço
2026-11-17 01:13:21,373 | src.core.model | INFO | Criando features por nota fiscal
2026-11-17 01:13:21,383 | src.core.model | INFO | Criando features climáticas
2026-11-17 01:13:21,386 | src.core.model | INFO | Criando variáveis normalizadas
2026-11-17 01:13:21,397 | src.core.model | INFO | Criando features textuais
2026-11-17 01:13:21,421 | src.core.model | INFO | Criando flags analíticas
2026-11-17 01:13:21,430 | src.core.model | INFO | Feature engineering concluído | Novas colunas: 31


Colunas originais: 25
Colunas após FE: 56


#### 3.1 Novas Variáveis

In [19]:
df_model[[
    'produto',
    'produto_clean',
    'ticket_medio_item',
    'percentual_tributo',
    'produto_promocao',
    'qtd_itens_nota'
]].head()

,produto,produto_clean,ticket_medio_item,percentual_tributo,produto_promocao,qtd_itens_nota
0,ALHO A GRANEL kg,ALHO A GRANEL KG,8.290122,0.11227,0,82
1,PINO T 3 SAIDAS IL,PINO T 3 SAIDAS IL,8.290122,0.11227,0,82
2,MILHO QUERO 170G,MILHO QUERO 170G,8.290122,0.11227,0,82
3,BEB VERY 1250G SALAD,BEB VERY 1250G SALAD,8.290122,0.11227,0,82
4,ARR T JOAO LF T1 5kg,ARR T JOAO LF T1 5KG,8.290122,0.11227,0,82


#### 3.2 Sample

In [20]:
novas_colunas = [

    col for col in df_model.columns

    if col not in df.columns
]

novas_colunas

['ano',
 'mes',
 'dia',
 'hora',
 'dia_ano',
 'semana_ano',
 'fim_semana',
 'ticket_medio_item',
 'percentual_tributo',
 'preco_por_quantidade',
 'frequencia_produto',
 'frequencia_categoria',
 'frequencia_supermercado',
 'desvio_preco_produto',
 'percentual_desvio_preco',
 'produto_promocao',
 'qtd_itens_nota',
 'qtd_categorias_nota',
 'amplitude_termica',
 'chuva_intensa',
 'preco_unitario_scaled',
 'preco_total_scaled',
 'quantidade_scaled',
 'valor_total_nota_scaled',
 'temperatura_media_scaled',
 'chuva_mm_scaled',
 'produto_clean',
 'produto_tamanho_texto',
 'compra_grande',
 'produto_caro',
 'produto_barato']

#### 3.3 Promoções Detectadas

In [21]:
df_model[
    df_model['produto_promocao'] == 1
][[
    'produto',
    'preco_unitario',
    'percentual_desvio_preco'
]].head(20)

,produto,preco_unitario,percentual_desvio_preco
16,FANDANGOS 45G QJO,2.69,-0.154088
17,FANDANGOS 45G QJO,2.69,-0.154088
19,P AL WYDA 30X4MT,2.99,-0.167131
57,POPCOR YOKI 100G BAC,3.98,-0.168234
109,BEB NESQUIK 200ML MO,2.29,-0.276207
129,MORANGO FRES 250G,7.89,-0.160192
132,LIMAO THAITI kg,1.99,-0.412979
134,TOMATE ITALIANO kg,2.99,-0.505422
141,FANDANGOS 45G QJO,2.69,-0.154088
147,AMIDO M MAIZENA 200G,3.99,-0.210943


#### 3.4 Correlação Novas

In [22]:
df_model.corr(
    numeric_only=True
)

,temperatura_max,temperatura_min,temperatura_media,chuva_mm,dia_chuvoso,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,quantidade,...,preco_unitario_scaled,preco_total_scaled,quantidade_scaled,valor_total_nota_scaled,temperatura_media_scaled,chuva_mm_scaled,produto_tamanho_texto,compra_grande,produto_caro,produto_barato
temperatura_max,1.000000,0.749939,0.919932,-0.022684,-0.025551,-0.197539,-0.085985,-0.048378,6.719025e-02,0.047651,...,0.008818,0.043728,0.047651,-0.085985,0.919932,-0.022684,-4.737234e-02,-0.056688,-0.009757,0.011858
temperatura_min,0.749939,1.000000,0.929710,0.268224,0.403509,-0.224198,-0.122571,-0.066328,5.197794e-04,0.051024,...,0.000499,0.035675,0.051024,-0.122571,0.929710,0.268224,-6.228524e-02,-0.113545,0.002247,0.028241
temperatura_media,0.919932,0.929710,1.000000,0.151191,0.222979,-0.250268,-0.125747,-0.077849,4.331245e-02,0.052715,...,0.008190,0.047296,0.052715,-0.125747,1.000000,0.151191,-5.371508e-02,-0.125713,0.000202,0.022071
chuva_mm,-0.022684,0.268224,0.151191,1.000000,0.322087,-0.123268,-0.065921,-0.035777,-4.363290e-02,0.028841,...,0.015696,0.035583,0.028841,-0.065921,0.151191,1.000000,-5.254830e-03,0.057586,0.025492,0.009225
dia_chuvoso,-0.025551,0.403509,0.222979,0.322087,1.000000,-0.090302,-0.039699,-0.003028,-6.978193e-02,0.038582,...,-0.022387,0.002545,0.038582,-0.039699,0.222979,0.322087,-5.209496e-02,-0.188006,0.006417,0.042989
qtd_total_nota,-0.197539,-0.224198,-0.250268,-0.123268,-0.090302,1.000000,0.934965,0.895038,-1.615576e-01,-0.021629,...,-0.036154,-0.041345,-0.021629,0.934965,-0.250268,-0.123268,-2.432480e-01,0.564164,-0.017608,-0.014576
valor_total_nota,-0.085985,-0.122571,-0.125747,-0.065921,-0.039699,0.934965,1.000000,0.954783,-1.846779e-01,0.019486,...,0.017073,0.036229,0.019486,1.000000,-0.125747,-0.065921,-2.382417e-01,0.649780,0.029335,-0.031259
valor_total_tributos,-0.048378,-0.066328,-0.077849,-0.035777,-0.003028,0.895038,0.954783,1.000000,-2.878268e-01,0.008196,...,0.021267,0.037445,0.008196,0.954783,-0.077849,-0.035777,-2.513525e-01,0.653451,0.022161,-0.008719
cod_produto,0.067190,0.000520,0.043312,-0.043633,-0.069782,-0.161558,-0.184678,-0.287827,1.000000e+00,0.054134,...,-0.036307,-0.017512,0.054134,-0.184678,0.043312,-0.043633,2.655810e-02,-0.124002,-0.035131,0.019344
quantidade,0.047651,0.051024,0.052715,0.028841,0.038582,-0.021629,0.019486,0.008196,5.413397e-02,1.000000,...,-0.126128,0.251416,1.000000,0.019486,0.052715,0.028841,6.701180e-02,0.004930,-0.078861,0.259645


In [23]:
toc = time.time()
print(f"\n\033[33mEtapa 3 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 3 | Tempo: (00:00:00) [17/11/2026 01:13:21 - 17/11/2026 01:13:21]


### 4. Market Basket Analysis

Nesta etapa são identificadas relações entre produtos e categorias frequentemente comprados em conjunto utilizando técnicas de associação.

In [24]:
tic = time.time()

In [25]:
# ==========================================================
# MARKET BASKET ANALYSIS
# ==========================================================

basket_results = run_market_basket_analysis(

    df=df_model,

    min_support=0.01,

    metric='lift',

    min_threshold=1.2,

    max_len=3,

    top_n_rules=20
)

2026-11-17 01:13:21.674 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-17 01:13:21,719 | src.core.model | INFO | Iniciando Market Basket Analysis
2026-11-17 01:13:21,734 | src.core.model | INFO | Realizando limpeza dos dados
2026-11-17 01:13:21,740 | src.core.model | INFO | Selecionando produtos mais relevantes
2026-11-17 01:13:21,751 | src.core.model | INFO | Construindo basket matrix
2026-11-17 01:13:21,761 | src.core.model | INFO | Executando FP-Growth
2026-11-17 01:13:22,185 | src.core.model | INFO | Gerando regras de associação
2026-11-17 01:13:22,986 | src.core.model | INFO | Calculando frequência dos produtos
2026-11-17 01:13:22,990 | src.core.model | INFO | Calculando métricas resumidas
2026-11-17 01:13:22,996 | src.core.model | INFO | Market Basket Analysis concluído


#### 4.1 Resumo Geral

In [26]:
basket_results['metrics_summary']

{'total_notas': 221,
 'total_produtos': 150,
 'total_regras': 71684,
 'media_lift': np.float64(9.5092),
 'media_confidence': np.float64(0.447),
 'media_support': np.float64(0.0167)}

#### 4.2 Regras Mais Fortes

In [27]:
basket_results['top_rules'][[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]]

,antecedents,consequents,support,confidence,lift
0,"BEB GATORADE 500ML, AGUA COCO SOCO 200ML",AG COC SOCOCO 200ML,0.013575,1.0,44.200000
1,"BEB LAC DANONE 250ML, MILHO QUERO 170G",ENERG RED BULL 250ML,0.013575,1.0,44.200000
2,"CEBOLA KG, BEB LAC TODD 200ML",ENERG RED BULL 250ML,0.013575,1.0,44.200000
3,"SALG ELMA CHIPS 35G, LEITE COND PIRA 395G",ENERG RED BULL 250ML,0.013575,1.0,44.200000
4,"LEITE COND PIRA 395G, BEB LAC TODD 200ML",ENERG RED BULL 250ML,0.013575,1.0,44.200000
5,"BEB LAC DANONE 250ML, BEB LAC TODD 200ML",ENERG RED BULL 250ML,0.013575,1.0,44.200000
6,"SALG ELMA CHIPS 35G, BEB LAC TODD 200ML",ENERG RED BULL 250ML,0.013575,1.0,44.200000
7,"REFRIG COCA COLA 2L, BEB LAC DANONE 250ML",ENERG RED BULL 250ML,0.013575,1.0,44.200000
8,"REFRIG COCA COLA 2L, SALG ELMA CHIPS 35G",ENERG RED BULL 250ML,0.013575,1.0,44.200000
9,AG COC SOCOCO 200ML,"BEB GATORADE 500ML, AGUA COCO SOCO 200ML",0.013575,0.6,44.200000


#### 4.3 Itemsets Frequentes

In [28]:
basket_results['frequent_items']

,support,itemsets
0,0.135747,frozenset({ENERG MONST 473ML})
1,0.135747,frozenset({CEBOLA KG})
2,0.131222,frozenset({LEITE TIROL TP 1L})
3,0.117647,frozenset({BEB LAC YOPRO 250ML})
4,0.117647,frozenset({DROPS HALLS 28G})
...,...,...
13561,0.013575,"frozenset({CAFE MELITTA 250G, ENERG MONSTER 47..."
13562,0.013575,"frozenset({MAC NISSIN 99G, CAFE MELITTA 250G})"
13563,0.013575,"frozenset({SALG ELMA CHIPS 35G, CAFE MELITTA 2..."
13564,0.013575,"frozenset({CAFE MELITTA 250G, MAC RENATA 85G})"


#### 4.4 Produtos Mais Frequentes

In [29]:
basket_results['product_frequency'].head(20)

,produto,frequencia
0,BEB LAC YOPRO 250ML,58
1,ENERG MONST 473ML,47
2,BEB LAC DANONE 250ML,47
3,SUCO MAIS MARACUJ 1L,46
4,DROPS HALLS 28G,43
5,LEITE TIROL TP 1L,42
6,CREME LTE LIDER 200G,42
7,AGUA COCO SOCO 200ML,42
8,MAC NISSIN 99G,41
9,BEB NESQUIK 200ML MO,36


#### 4.5 Regras Muito Fortes

In [30]:
basket_results['rules'][
    basket_results['rules']['lift'] >= 3
][[
    'antecedents',
    'consequents',
    'lift',
    'confidence'
]]

,antecedents,consequents,lift,confidence
0,"BEB GATORADE 500ML, AGUA COCO SOCO 200ML",AG COC SOCOCO 200ML,44.2,1.000000
1,"BEB LAC DANONE 250ML, MILHO QUERO 170G",ENERG RED BULL 250ML,44.2,1.000000
2,"CEBOLA KG, BEB LAC TODD 200ML",ENERG RED BULL 250ML,44.2,1.000000
3,"SALG ELMA CHIPS 35G, LEITE COND PIRA 395G",ENERG RED BULL 250ML,44.2,1.000000
4,"LEITE COND PIRA 395G, BEB LAC TODD 200ML",ENERG RED BULL 250ML,44.2,1.000000
...,...,...,...,...
70996,CAFE MELITTA 250G,"CREME LTE LIDER 200G, LEITE TIROL TP 1L",3.0,0.230769
70997,"ENERG MONST 473ML, DROPS HALLS 28G",CENOURA KG,3.0,0.230769
70998,DETERG LIMPOL 500ML,"LIMAO KG, ALHO KG",3.0,0.230769
70999,REFR COCA COLA 200ML,CENOURA KG,3.0,0.230769


In [31]:
toc = time.time()
print(f"\n\033[33mEtapa 4 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 4 | Tempo: (00:00:01) [17/11/2026 01:13:21 - 17/11/2026 01:13:23]


### 5. NLP & Embeddings

Nesta etapa são gerados embeddings semânticos dos produtos para análises de similaridade, agrupamento e recomendação.

In [32]:
tic = time.time()

In [33]:
df['produto_clean']

KeyError: 'produto_clean'

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import (
    SentenceTransformer
)

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

In [ ]:
embeddings = model.encode(
    df['produto'].unique()
)

In [ ]:
cosine_similarity(...)

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 5 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 6. IA Generativa e Insights Automáticos

Nesta etapa são exploradas abordagens de LLMs para geração automática de insights, explicações estatísticas e interpretação dos padrões de consumo.

In [ ]:
tic = time.time()

In [ ]:
def generate_insight(...):

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 6 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 8. Próximos Passos

- Sistemas de recomendação
- Forecast temporal
- Detecção de anomalias
- RAG sobre dados financeiros
- IA contextual com clima e inflação

## 3. Tempo Decorrido

In [ ]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")